In [2]:
import requests

proxies = {
    'http': 'http://127.0.0.1:10809',
    'https': 'http://127.0.0.1:10809',
}

response = requests.get('https://www.google.com', proxies=proxies)
print(response.status_code)


200


In [3]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import json

SCOPES = ["https://www.googleapis.com/auth/youtube.readonly"]

def get_authenticated_service():
    flow = InstalledAppFlow.from_client_secrets_file(r"C:\02Programmer\02Proj\PyVSCode\client_secret.json", SCOPES)
    credentials = flow.run_local_server(port=8080)
    return build("youtube", "v3", credentials=credentials)

def get_watch_later_playlist_id(youtube):
    playlists = youtube.playlists().list(
        part="snippet",
        mine=True,
        maxResults=50
    ).execute()

    for item in playlists.get("items", []):
        if item["snippet"]["title"].lower() == "watch later":
            return item["id"]
    return None

def get_watch_later_videos(youtube, playlist_id):
    videos = []
    next_page_token = None

    while True:
        response = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()

        for item in response["items"]:
            title = item["snippet"]["title"]
            video_id = item["snippet"]["resourceId"]["videoId"]
            url = f"https://www.youtube.com/watch?v={video_id}"
            videos.append({"title": title, "url": url})

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return videos

if __name__ == "__main__":
    youtube = get_authenticated_service()
    playlist_id = get_watch_later_playlist_id(youtube)

    if not playlist_id:
        print("❌ 没有找到 Watch Later 列表")
    else:
        videos = get_watch_later_videos(youtube, playlist_id)
        print(f"✅ 共找到 {len(videos)} 条视频")
        with open("watch_later.json", "w", encoding="utf-8") as f:
            json.dump(videos, f, ensure_ascii=False, indent=2)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=289899744157-8adg7a9v58h9lgnjmgmu0g2ok86gr7hk.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.readonly&state=uwuYttHD8refBHGhgrHOhNn4fqkWK7&access_type=offline


TimeoutError: [WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。

In [2]:
import requests
from bs4 import BeautifulSoup

# 获取网页内容
url = "https://www.youtube.com/playlist?list=WL"
response = requests.get(url)

# 检查请求是否成功
if response.status_code == 200:
    soup = BeautifulSoup(response.text, "lxml")

    # 提取所有目标类中的href属性
    links = soup.find_all("a", class_="yt-simple-endpoint style-scope yt-formatted-string")

    # 输出所有的href
    for link in links:
        href = link.get("href")
        if href:
            print(href)
else:
    print(f"请求失败，状态码: {response.status_code}")


In [ ]:
from bs4 import BeautifulSoup

# 读取本地HTML文件
with open("稍后观看 - YouTube.html", "r", encoding="utf-8") as file:
    html_content = file.read()

# 解析HTML
soup = BeautifulSoup(html_content, "lxml")

# 提取所有目标类（yt-simple-endpoint style-scope yt-formatted-string）中的href属性
links = soup.find_all("a", class_="yt-simple-endpoint style-scope yt-formatted-string")

# 输出所有的href
for link in links:
    href = link.get("href")
    if href:
        print(href)


In [4]:
from urllib.parse import urlparse

def extract_channel_name(url):
    path = urlparse(url).path.strip("/")
    if path.startswith("@"):
        return path.split("/")[0][1:]  # 移除 @
    else:
        return path.split("/")[1] if "/" in path else path
    
extract_channel_name("https://www.youtube.com/@ABCD/videos")

'ABCD'